In [ ]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

### Область работы 1 (библиотеки)

In [ ]:
# Данный блок в области 1 НЕ выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
#
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
#
# pip install

In [89]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import category_encoders as ce
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder, OneHotEncoder, TargetEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import PolynomialFeatures
from scipy import stats
from sklearn.compose import ColumnTransformer, make_column_transformer

from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.model_selection import GridSearchCV,train_test_split

from sklearn.linear_model import Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR


from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.metrics import mean_absolute_percentage_error,mean_squared_error,r2_score,make_scorer
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import PolynomialFeatures

from sklearn.metrics import mean_squared_log_error



### Область работы 2 (поиск  модели .... )

In [158]:
# Данный блок(и) НЕ выполняются преподавателем в области 2
# блок(и) предназначены для поиска лучшей модели 
# должен быть понятен и очевиден отбор параметров модели
# 
# Запрещается размещать данные блоки за пределами обасти 2
# Все блоки данной области должны быть выполнены
#
# ЗАПРЕЩАЕТСЯ ИСПОЛЬЗОВАТЬ ТЕСТОВЫЙ НАБОР
#
# Путь к тренировочному набору
# 

path_train = 'train.csv' # содержит только имя файла, без имен папок !!!
df=pd.read_csv(path_train)

In [92]:
df

,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,Adidas,Nylon,Large,3.0,No,NaN,Backpack,Black,13.7,29.0
1,Adidas,Nylon,Medium,1.0,No,Yes,Tote,Pink,6.9,68.0
2,Nike,Polyester,Small,4.0,Yes,No,Messenger,Gray,18.3,84.0
3,Adidas,Leather,Small,10.0,No,Yes,Messenger,Blue,26.5,34.0
4,Adidas,Nylon,Medium,9.0,Yes,No,Messenger,Black,17.9,78.0
...,...,...,...,...,...,...,...,...,...,...
19995,Adidas,Canvas,Large,9.0,No,No,Backpack,Red,15.0,91.0
19996,Puma,Polyester,Small,10.0,Yes,No,Messenger,Pink,18.9,59.0
19997,Nike,Canvas,Small,9.0,Yes,NaN,Tote,Red,16.8,40.0
19998,Adidas,Leather,Small,6.0,No,No,Backpack,Gray,29.5,40.0


In [159]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand                 20000 non-null  object 
 1   Material              18961 non-null  object 
 2   Size                  19003 non-null  object 
 3   Compartments          19039 non-null  float64
 4   Laptop Compartment    19033 non-null  object 
 5   Waterproof            19013 non-null  object 
 6   Style                 18998 non-null  object 
 7   Color                 18976 non-null  object 
 8   Weight Capacity (kg)  19003 non-null  float64
 9   Price                 20000 non-null  float64
dtypes: float64(3), object(7)
memory usage: 1.5+ MB


In [160]:
df.describe()

,Compartments,Weight Capacity (kg),Price
count,19039.000000,19003.000000,20000.0000
mean,5.530543,17.535142,82.0595
std,2.870658,7.488566,39.1298
min,1.000000,5.000000,15.0000
25%,3.000000,11.100000,48.0000
50%,6.000000,17.500000,82.0000
75%,8.000000,24.000000,115.0000
max,10.000000,30.000000,150.0000


In [161]:
X = df.drop(columns=["Price"])
y = df["Price"]

In [162]:
categorical_features = [
    "Brand",
    "Material",
    "Size",
    "Style",
    "Color",
    "Laptop Compartment",
    "Waterproof"
]

numeric_features = [
    "Compartments",
    "Weight Capacity (kg)"
]


In [163]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [164]:
CT = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [188]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [166]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(0, y_pred)  
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

РИДЖ

In [167]:
model_ridge = Pipeline(steps=[
    ("preprocessor", CT),
    ("model", Ridge(alpha=0.01))
])

model_ridge.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Compartments',
                                                   'Weight Capacity (kg)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Brand', 'Material', 'Size',
                                                   'Style', 'Color',
                                                   'Laptop Compartment',
                                                   'Waterproof'])])),
                ('model', Ridge(alpha=0.01))])

In [168]:
y_pred_ridge = model_ridge.predict(X_test)

In [169]:
r2_score(y_test,y_pred_ridge)

0.00015471135379852008

In [170]:
ridge_rmsle = rmsle(y_test, y_pred_ridge)
print("Ridge RMSLE:", ridge_rmsle)

Ridge RMSLE: 0.5948074807876049


ГРАДИЕНТНЫЙ БУСТИНГ

In [171]:
model_GB = Pipeline(steps=[
    ("preprocessor", CT),
    ("model", GradientBoostingRegressor())
])

model_GB.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Compartments',
                                                   'Weight Capacity (kg)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Brand', 'Material', 'Size',
                                                   'Style', 'Color',
                                                   'Laptop Compartment',
                                                   'Waterproof'])])),
                ('model', GradientBoostingRegressor())])

In [172]:
y_pred_GB = model_GB.predict(X_test)

In [173]:
r2_score(y_test,y_pred_GB)

-0.0021553421587554578

In [174]:
gbr_rmsle = rmsle(y_test, y_pred_GB)

print("Gradient Boosting RMSLE:", gbr_rmsle)

Gradient Boosting RMSLE: 0.5949655592607644


РАНДОМ ФОРЕСТ

In [175]:
model_RF = Pipeline(steps=[
    ("preprocessor", CT),
    ("model", RandomForestRegressor(
        n_estimators=1000,
        n_jobs=-1
    ))
])

model_RF.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Compartments',
                                                   'Weight Capacity (kg)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Brand', 'Material', 'Size',
                                                   'Style', 'Color',
                                                   'Laptop Compartment',
                                                   'Waterproof'])])),
                ('model', RandomForestRegressor(n_estimators=1000, n_jobs=-1))])

In [176]:
y_pred_RF = model_RF.predict(X_test)

In [177]:
r2_score(y_test,y_pred_RF)

-0.04724976507116674

In [178]:
rf_rmsle = rmsle(y_test, y_pred_RF)
print("Random Forest RMSLE:", rf_rmsle)

Random Forest RMSLE: 0.6015710286657201


SVR

In [179]:
model_svr = Pipeline(steps=[
    ("preprocessor", CT),
    ("model", SVR())
])

model_svr.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Compartments',
                                                   'Weight Capacity (kg)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Brand', 'Material', 'Size',
                                                   'Style', 'Color',
                                                   'Laptop Compartment',
                                                   'Waterproof'])])),
                ('model', SVR())])

In [180]:
y_pred_svr = model_svr.predict(X_test)

In [181]:
r2_score(y_test,y_pred_svr)

-0.003663353842236594

In [182]:
svr_rmsle = rmsle(y_test, y_pred_svr)
print("SVR RMSLE:", svr_rmsle)

SVR RMSLE: 0.5946513952007084


In [183]:
model_lasso = Pipeline(steps=[
    ("preprocessor", CT), 
    ("model", Lasso(alpha=0.01))  
])

model_lasso.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Compartments',
                                                   'Weight Capacity (kg)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Brand', 'Material', 'Size',
                                                   'Style', 'Color',
                                                   'Laptop Compartment',
                                                   'Waterproof'])])),
                ('model', Lasso(alpha=0.01))])

In [184]:
y_pred_lasso = model_lasso.predict(X_test)

In [185]:
r2_score(y_test,y_pred_lasso)

0.00017662454441780273

In [186]:
lasso_rmsle = rmsle(y_test, y_pred_lasso)
print("SVR RMSLE:", lasso_rmsle)

SVR RMSLE: 0.5948087490463991


In [187]:
results = pd.DataFrame({
    "Model": ["Ridge", "Gradient Boosting", "Random Forest", "SVR", "Lasso"],
    "RMSLE": [ridge_rmsle, gbr_rmsle, rf_rmsle, svr_rmsle, lasso_rmsle]
})

print(results.sort_values("RMSLE"))


               Model     RMSLE
3                SVR  0.594651
0              Ridge  0.594807
4              Lasso  0.594809
1  Gradient Boosting  0.594966
2      Random Forest  0.601571
